# Step 1: A2A Agent Olympics (MVP)
Generalist vs Supervisor+Specialists, built in a single VS Code Jupyter notebook.

What you get
- 4 local A2A-ish agent servers (Generalist, Math, Schema, Cartoon)
- A Referee that discovers agents via `/.well-known/agent.json`, then routes tasks to specialists
- Objective scoring for tasks 1 and 2, plus a fun task 3 for audience voting

Notes
- This MVP implements a minimal subset of the A2A patterns: Agent Card discovery + JSON-RPC `message/send` + `tasks/get`.
- It follows the published examples for the Agent Card path `/.well-known/agent.json` and the JSON-RPC `message/send` shape.


## 1) Write the local agent servers
Each agent exposes:
- `GET /.well-known/agent.json` (Agent Card)
- `POST /a2a` JSON-RPC with:
  - `message/send`
  - `tasks/get`

Ports:
- Generalist: 8100
- Math: 8101
- Schema: 8102
- Cartoon: 8103


In [1]:
from dotenv import load_dotenv
load_dotenv()

import os, sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Could not find repo root (pyproject.toml not found).")

REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR   = REPO_ROOT / "src"
APP_DIR   = SRC_DIR / "demos" / "a2a_olympics"

# Notebook imports (so *this* kernel can import a2a.core)
sys.path.insert(0, str(SRC_DIR.resolve()))
sys.path.insert(0, str(APP_DIR.resolve()))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_DIR  :", SRC_DIR)
print("APP_DIR  :", APP_DIR)

from a2a.core import (
    AgentRegistry, fetch_agent_card, A2AHttpAgent, Task,
    start_uvicorn, wait_http_ok, tail, stop_proc
)

HOST = "127.0.0.1"
ROOT_DIR = REPO_ROOT  # ✅ important

agents = {
    "generalist": ("generalist_agent:app", 8100),
    "math":       ("math_agent:app", 8101),
    "schema":     ("schema_agent:app", 8102),
    "story":      ("story_agent:app", 8103),
}

# ✅ This is the real missing piece: PYTHONPATH for the child process
child_env = dict(os.environ)
child_env["PYTHONPATH"] = str(SRC_DIR.resolve()) + (
    (os.pathsep + child_env["PYTHONPATH"]) if child_env.get("PYTHONPATH") else ""
)

procs = {}
for name, (spec, port) in agents.items():
    url = f"http://{HOST}:{port}/.well-known/agent.json"
    try:
        wait_http_ok(url, timeout_s=1)
        print(f"✅ {name} already running on {port}")
        continue
    except Exception:
        pass

    print(f"▶ starting {name} on {port} ...")
    p = start_uvicorn(spec, HOST, port, cwd=ROOT_DIR, env=child_env, app_dir=APP_DIR)
    procs[name] = p

    try:
        wait_http_ok(url, timeout_s=20)
        print(f"✅ {name} up: {url}")
    except Exception as e:
        print(f"❌ {name} failed: {e}")
        print(tail(p, n=120))
        stop_proc(p)
        raise



REPO_ROOT: /Users/douglasdaly/Documents/GitHub/Generative-AI
SRC_DIR  : /Users/douglasdaly/Documents/GitHub/Generative-AI/src
APP_DIR  : /Users/douglasdaly/Documents/GitHub/Generative-AI/src/demos/a2a_olympics
▶ starting generalist on 8100 ...
✅ generalist up: http://127.0.0.1:8100/.well-known/agent.json
▶ starting math on 8101 ...
✅ math up: http://127.0.0.1:8101/.well-known/agent.json
▶ starting schema on 8102 ...
✅ schema up: http://127.0.0.1:8102/.well-known/agent.json
▶ starting story on 8103 ...
✅ story up: http://127.0.0.1:8103/.well-known/agent.json


# 2. Discover agents

## Perform task

In [ ]:
from tasks import (
    TASK1_PROMPT, TASK1_CONTRACT, score_task1, 
    TASK2_PROMPT, TASK2_CONTRACT, score_task2, 
    TASK3_PROMPT, TASK3_CONTRACT, score_task3)
from datetime import datetime as dt
from pathlib import Path

# Used to save stories to files
def story_text(res) -> str:
    # Reuse your extractor if you have it; otherwise:
    if hasattr(res, "output") and isinstance(res.output, str):
        return res.output.strip()
    # fallback: stringify
    return str(getattr(res, "output", res)).strip()

def save_story(res, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    txt = story_text(res)
    path.write_text(txt, encoding="utf-8")
    return path

# Define tasks
tasks: list[Task] = [
    Task(
        id="T1",
        kind="math_transcendental_roots",
        description="Find the 10 real solutions with smallest |x| to sin(x) = x/20.",
        prompt=TASK1_PROMPT,
        needed_skills=["math.solve.real_roots", "math.verify.residuals"],  # or whatever your agent card advertises
        score_fn=score_task1,
        contract=TASK1_CONTRACT,
    ),
    Task(
    id="T2",
    description="Extract a structured JSON object from a log line per schema.",
    prompt=TASK2_PROMPT,
    needed_skills=["schema.extract", "schema.validate"],
    score_fn=score_task2,
    contract=TASK2_CONTRACT,
    ),
    Task(
    id="T3",
    description="Write a 1950s hard-boiled detective noir story in EXACTLY 3 chapters.",
    prompt=TASK3_PROMPT,
    needed_skills=["story.parse", "story.check"],
    score_fn=score_task3,
    contract=TASK3_CONTRACT,
    )
]


# 2) Build registry of clients by type at known servers.
agent_urls = {
    "generalist": "http://127.0.0.1:8100",
    "math":       "http://127.0.0.1:8101",
    "schema":     "http://127.0.0.1:8102",
    "story":      "http://127.0.0.1:8103",
}
cards = [fetch_agent_card(u) for u in agent_urls.values()]
reg = AgentRegistry(cards)

general_card = reg.by_name["generalist"]  # or match by url/name depending on your card
general_agent = A2AHttpAgent(general_card, timeout_s=90)


# 3) Run tasks - the spec_card/spec_agent is the first card/agent that advertises the needed skills.
for t in tasks:
    spec_card = reg.get_agent(t.needed_skills, fallback_name=general_card.name)
    spec_agent = A2AHttpAgent(spec_card, timeout_s=180)   # stories can be slow
    print("\n" + "="*90)
    print(f"{t.id}: {t.description}")
    print("General:", reg.describe_agent(general_card))
    print("Spec:   ", reg.describe_agent(spec_card))

    gen_res  = general_agent.invoke(t.prompt)
    spec_res = spec_agent.invoke(t.prompt)

    # For story task, save outputs to files
    if t.needed_skills[0].startswith("story."):
      stamp = dt.now().strftime("%Y%m%d_%H%M%S")
      outdir = ROOT_DIR / "results" / "a2a_olympics" / "runs" / f"task3_{stamp}"
      gen_path  = save_story(gen_res,  outdir / "story_generalist.txt")
      spec_path = save_story(spec_res, outdir / "story_specialist.txt")
      print("Saved:", gen_path, spec_path)

    gen_score  = t.score_fn(gen_res)
    spec_score = t.score_fn(spec_res)

    print("\nGEN SCORE:", gen_score["score"], "\nDetails:", gen_score["summary"])
    print("SPEC SCORE:", spec_score["score"], "\nDetails:", spec_score["summary"])




T1: Find the 10 real solutions with smallest |x| to sin(x) = x/20.
General: generalist @ http://127.0.0.1:8100
skills: general.chat
card_sha256: 8fb0cb00ca...
Spec:    math @ http://127.0.0.1:8101
skills: math.solve.real_roots, math.verify.residuals
card_sha256: 5c3a7a3ffc...

GEN SCORE: 0.0 
Details: {'residuals': [0.2746409233732041, 0.7853981633979736, 0.4712388980384686, 0.15707963267948954, 0.15707963267948954, 0.4712388980384686, 0.7853981633979736, 0.2746409233732041, 0.6283185307179591, 0.6283185307179591]}
SPEC SCORE: 10.0 
Details: {'residuals': [2.1516122217235534e-13, 3.4849900742983664e-13, 4.2577052994374753e-14, 5.362377208939506e-14, 2.1094237467877974e-14, 2.1094237467877974e-14, 5.362377208939506e-14, 4.2577052994374753e-14, 3.4849900742983664e-13, 2.1516122217235534e-13]}

T2: Extract a structured JSON object from a log line per schema.
General: generalist @ http://127.0.0.1:8100
skills: general.chat
card_sha256: 8fb0cb00ca...
Spec:    schema @ http://127.0.0.1:8102

In [3]:
# Stop processes when done
for name, p in procs.items():
    print(f"■ stopping {name} ...")
    stop_proc(p)

■ stopping generalist ...
■ stopping math ...
■ stopping schema ...
■ stopping story ...
